# Streaming responses from OpenAI GPT models

This example demonstrates how to stream the response of an OpenAI GPT model from a Jupyter notebook. Instead of waiting for the complete answer, the notebook prints each fragment as the model produces it and reports the time to first token.

## 1. Setup

Set your OpenAI API key in the `OPENAI_API_KEY` environment variable before running the notebook.

In [1]:
%pip install openai

In [4]:
import os
import time
from getpass import getpass
from openai import OpenAI

# Ask for the API key interactively if not defined as env variable
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

client = OpenAI()

def query_model(prompt: str,
                model: str = "gpt-6-luna",
                max_tokens: int = 1024,
                temperature: float = 0) -> str:
    """Stream the response of an OpenAI model."""
    start = time.perf_counter()
    first_token = None
    chunks = []
    usage = None
    model_id = model

    stream = client.responses.create(
        model=model,
        input=prompt,
        max_output_tokens=max_tokens,
        reasoning={"effort": "none"},  # temperature requires no reasoning
        temperature=temperature,
        stream=True,
    )
    for event in stream:
        if event.type == "response.output_text.delta":
            if first_token is None:
                first_token = time.perf_counter() - start
            print(event.delta, end="", flush=True)
            chunks.append(event.delta)
        elif event.type == "response.completed":
            usage = event.response.usage
            model_id = event.response.model
    latency = time.perf_counter() - start
    print()

    ttft = first_token if first_token is not None else latency
    print(f"\tModel: {model_id}")
    print(f"\tTime to first token: {ttft:.3f} seconds")
    print(f"\tLatency: {latency:.3f} seconds")
    if usage is not None:
        print(f"\tInput tokens: {usage.input_tokens}")
        print(f"\tOutput tokens: {usage.output_tokens}")
        print(f"\tTotal tokens: {usage.total_tokens}")

    return "".join(chunks).strip()

In [5]:
prompt = "How many tokens are in your context window?"
print("User:", prompt)
print("AI: ", end="", flush=True)
query_model(prompt)

User: How many tokens are in your context window?
AI: I don’t have access to the exact context-window limit for this conversation.
	Model: gpt-6-luna
	Time to first token: 2.323 seconds
	Latency: 2.582 seconds
	Input tokens: 15
	Output tokens: 19
	Total tokens: 34


'I don’t have access to the exact context-window limit for this conversation.'